In [ ]:
import os.path

from datasets import load_from_disk, load_dataset
from torch.utils.data import DataLoader, ConcatDataset
from kaamba_repo.net.models.tamba import GazePredictor
from kaamba_repo.utils.loss_functions import gaussian_nll
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torchvision.utils import draw_bounding_boxes, draw_keypoints, draw_segmentation_masks
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as F


In [5]:
import kaamba

In [8]:
kaamba.scripts.train_on_the_fly("C:/Users/saphi/PycharmProjects/thesis/kaamba_repo/src/kaamba/kaamba_dataset/stimuli/Goettingen/metadata_Goettingen.parquet")


⚠️  CUDA not available, using CPU
TRAINING WITH ON-THE-FLY SEQUENCE GENERATION
Dataset: C:/Users/saphi/PycharmProjects/thesis/kaamba_repo/src/kaamba/kaamba_dataset/stimuli/Goettingen/metadata_Goettingen.parquet
Device: cpu
Batch size: 32
Workers: 4
Context length: 32
Stride: 1

Creating data loader...
Goettingen
Initializing model...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 90,536,036

TRAINING

--- Epoch 1/3 ---


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "C:\Users\saphi\PycharmProjects\thesis\.venv\lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "C:\Users\saphi\PycharmProjects\thesis\.venv\lib\site-packages\torch\utils\data\_utils\fetch.py", line 35, in fetch
    data.append(next(self.dataset_iter))
  File "C:\Users\saphi\PycharmProjects\thesis\kaamba_repo\src\kaamba\utils\on_the_fly_dataset.py", line 97, in __iter__
    data = self.data.collect() if self.lazy else self.data
  File "C:\Users\saphi\PycharmProjects\thesis\.venv\lib\site-packages\polars\_utils\deprecation.py", line 97, in wrapper
    return function(*args, **kwargs)
  File "C:\Users\saphi\PycharmProjects\thesis\.venv\lib\site-packages\polars\lazyframe\opt_flags.py", line 326, in wrapper
    return function(*args, **kwargs)
  File "C:\Users\sa

In [160]:
import pymovements as pm
pm.DatasetLibrary.names()


['BSC',
 'BSCII',
 'ChineseReading',
 'CoLAGaze',
 'CodeComprehension',
 'CopCo',
 'DAEMONS',
 'DIDEC',
 'EMTeC',
 'ETDD70',
 'FakeNewsPerception',
 'GGTG',
 'Gaze4Hate',
 'GazeBase',
 'GazeBaseVR',
 'GazeGraph',
 'GazeOnFaces',
 'HBN',
 'IITB_HGC',
 'InteRead',
 'JuDo1000',
 'MECOL1W1',
 'MECOL1W2',
 'MECOL2W1',
 'MECOL2W2',
 'MouseCursor',
 'OneStop',
 'PoTeC',
 'PotsdamBingeRemotePVT',
 'PotsdamBingeWearablePVT',
 'Provo',
 'RaCCooNS',
 'SBSAT',
 'TECO',
 'ToyDataset',
 'ToyDatasetEyeLink',
 'UCL']

In [ ]:
dataset = "MECOL1W1"
dataset_paths = pm.DatasetPaths(root='data/')
pm_dataset = pm.Dataset(dataset, path=dataset_paths)

In [ ]:
pm_dataset.download()
pm_dataset.extract(remove_finished=True)
pm_dataset.scan()


In [ ]:
def plot(imgs, row_title=None, bbox_width=3, **imshow_kwargs):
    if not isinstance(imgs[0], list):
        # Make a 2d grid even if there's just 1 row
        imgs = [imgs]

    num_rows = len(imgs)
    num_cols = len(imgs[0])
    _, axs = plt.subplots(nrows=num_rows, ncols=num_cols, squeeze=False)
    for row_idx, row in enumerate(imgs):
        for col_idx, img in enumerate(row):
            boxes = None
            masks = None
            points = None
            if isinstance(img, tuple):
                img, target = img
                if isinstance(target, dict):
                    boxes = target.get("boxes")
                    masks = target.get("masks")
                elif isinstance(target, tv_tensors.BoundingBoxes):
                    boxes = target

                    # Conversion necessary because draw_bounding_boxes() only
                    # work with this specific format.
                    if tv_tensors.is_rotated_bounding_format(boxes.format):
                        boxes = v2.ConvertBoundingBoxFormat("xyxyxyxy")(boxes)
                elif isinstance(target, tv_tensors.KeyPoints):
                    points = target
                else:
                    raise ValueError(f"Unexpected target type: {type(target)}")
            img = F.to_image(img)
            if img.dtype.is_floating_point and img.min() < 0:
                # Poor man's re-normalization for the colors to be OK-ish. This
                # is useful for images coming out of Normalize()
                img -= img.min()
                img /= img.max()

            img = F.to_dtype(img, torch.uint8, scale=True)
            if boxes is not None:
                img = draw_bounding_boxes(img, boxes, colors="yellow", width=bbox_width)
            if masks is not None:
                img = draw_segmentation_masks(img, masks.to(torch.bool), colors=["green"] * masks.shape[0], alpha=.65)
            if points is not None:
                img = draw_keypoints(img, points, colors="red", radius=10)

            ax = axs[row_idx, col_idx]
            ax.imshow(img.permute(1, 2, 0).numpy(), **imshow_kwargs)
            ax.set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

    if row_title is not None:
        for row_idx in range(num_rows):
            axs[row_idx, 0].set(ylabel=row_title[row_idx])

    plt.tight_layout()

In [ ]:

orig_img = Image.open(Path("/kaamba_repo/kaamba_dataset/stimuli/Goettingen/arg_pisacowsmilk_id10_page_1_de.png"))
orig_img

In [ ]:
padded_imgs = [v2.Pad(padding=padding, padding_mode="edge")(orig_img) for padding in ([0,0, 10, 10],[0, 0, 50, 50])]
plot([orig_img] + padded_imgs)

In [ ]:
resized_imgs = [v2.Resize(size=size, max_size=675)(orig_img) for size in (256, 512)]
padded_image = [v2.Pad(padding=[0,0, 675 - orig_img.size[0], 675 - orig_img.size[1]], padding_mode="edge")(orig_img) for orig_img in resized_imgs]
padded_image[0]

In [ ]:
class MyCustomTransform(v2.Pad):
    def __init__(self, *args, **kwargs):
        super().__init__(padding = 0, *args, **kwargs)

    def forward(self, img):
        """
        Args:
            img (PIL Image or Tensor): Image to be padded.

        Returns:
            PIL Image or Tensor: Padded image.

        """
        print(
            f"I'm transforming an image of shape {img.shape} "

        )
        pad_vals = [0, 0, img.shape[2] - img.shape[2], img.shape[2] - img.shape[1]]
        return F.pad(img, pad_vals, self.fill, self.padding_mode)



In [ ]:
image.shape[2]

In [ ]:
screen_resolution = (1920, 1080)


padded_im = v2.Pad(padding=padding_val, padding_mode="edge")
padded_im

In [ ]:
from torchvision.io import decode_image
max_size = 512
padding_val = [0, 0, screen_resolution[0]- image.shape[2], screen_resolution[1]- image.shape[1]]
transform = v2.Compose([
            v2.Pad(padding=padding_val, padding_mode="edge"),
            v2.Resize(size = None, max_size = max_size), #ToDo just for testing purposes has to be adapted to the actual image size and model requirements max size
            MyCustomTransform(padding_mode="edge"),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # get normalized values for RGB channels (assuming image is in RGB format
        ])
image = decode_image(Path("/kaamba_repo/kaamba_dataset/stimuli/Goettingen/arg_pisacowsmilk_id10_page_1_de.png"))


In [ ]:
transforemd = transform(image)

In [ ]:
plot(transforemd)

In [ ]:
def _image_transform(image_path: Path, max_size=512) -> torch.Tensor:
        screen_resolution = (1920, 1080)  #SCREEN_RESOLUTION[self.datacollection_name] # for example (1920, 1080)  # Example screen resolution, adjust as needed
        image = decode_image(str(image_path))
        padding_val = [0, 0, screen_resolution[0]- image.shape[2], screen_resolution[1]- image.shape[1]]
        scaling_factor =  max_size / screen_resolution[0]
        transform = v2.Compose([
            v2.Pad(padding=padding_val, padding_mode="edge"),
            v2.Resize(size = None, max_size = max_size), #ToDo just for testing purposes has to be adapted to the actual image size and model requirements max size
            MyCustomTransform(padding_mode="edge"),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # get normalized values for RGB channels (assuming image is in RGB format
        ])

        return transform(image)

In [ ]:
max_size=512
screen_resolution = (1920, 1080)

scaling_factor

In [ ]:
1080 * scaling_factor

In [ ]:
trans = _image_transform(Path("/kaamba_repo/kaamba_dataset/stimuli/Goettingen/arg_pisacowsmilk_id10_page_1_de.png"))
plot([trans, orig_img])

In [ ]:
from torchvision.transforms.v2 import functional as F

In [ ]:

path = Path("/kaamba_repo/kaamba_dataset")

In [ ]:
img_with_paths = Path("data/MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_MultiplEYE_DE_DE_Goettingen_1_2026/multipleye_stimuli_experiment_de_de_1_with_img_paths.csv")
out_Dir = Path("data/MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_images_de_de_1")

In [ ]:
dataset = load_dataset(str(path))

In [ ]:
dataset = dataset.with_format("torch")

In [ ]:
target

In [ ]:
input = (next(iter(dataset["input_seq"][0])))


In [ ]:
input.float()

In [ ]:
%%sql


In [ ]:
for batch in loader:
    print(batch)



In [ ]:
import os
os.path.exists(path)

In [ ]:
for key in dataset.keys():
    print(key)

In [ ]:
print(next(iter(dataset["data"])))

In [ ]:
from datasets import load_dataset
dataset = load_dataset("data/MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_images_de_de_1")


In [ ]:
dataset["train"][0]

In [ ]:
from torch.utils.data import Dataset
dataset.features

In [ ]:
dataset["train"][0]["data"]

In [ ]:
from torchvision.io import decode_image
img = decode_image("data/MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_MultiplEYE_DE_DE_Goettingen_1_2026/stimuli_images_de_de_1/arg_pisacowsmilk_id10_page_1_de.png", mode="RGB")
img = img.flatten()